# Merger filing research and extraction

Aaron’s merger contribution: validate forms → load real filings → inspect structure → experiment with extraction → map to BNY fields and evaluation. See [the full report](../docs/merger_research.md). All outputs are candidate evidence; canonical fields remain null.

In [1]:
from pathlib import Path
import json
from collections import Counter
ROOT = Path.cwd() if (Path.cwd() / 'data/mergers').exists() else Path.cwd().parent
def read(name):
    return json.loads((ROOT / 'outputs/mergers' / name).read_text())
rows = read('extractions.json')
summary = read('summary.json')
print(json.dumps(summary, indent=2))

{
  "documents_requested": 35,
  "documents_processed": 35,
  "document_detection": {
    "merger_candidate": 26,
    "review_no_strong_merger_cue": 5,
    "review_exchange_offer_overlap": 4
  },
  "by_form": {
    "425": 5,
    "PREM14A": 5,
    "DEFM14A": 5,
    "SC 13E3": 5,
    "SC 13E3/A": 5,
    "S-4": 5,
    "S-4/A": 5
  },
  "documents_with_value_candidates": 19,
  "candidate_counts": {
    "cash_amount": 354,
    "conversion_ratio": 216,
    "record_date": 11
  },
  "documents_with_conflicting_values": 17,
  "processing_seconds_total": 30.389076830091653,
  "processing_seconds_median": 0.3557979160104878,
  "llm_tokens": 0,
  "llm_api_cost_usd": 0,
  "total_compute_cost_usd": null,
  "detection_f1": null,
  "event_level_accuracy": null,
  "discovery_recall": null,
  "metric_note": "Development sample; candidates are unverified. No independent gold labels or complete discovery denominator. Timing excludes network; local compute cost unmeasured."
}


## Form validation and metadata coverage

DEFM14A/PREM14A are merger-or-acquisition proxies; S-4 also covers exchanges; SC 13E3 covers certain going-private structures; 425 supplies communications. Merger 8-K discovery remains a gap because the existing 8-K track is conversion-filtered. See official SEC references in the report. Counts below are metadata, not verified merger events.

In [2]:
for r in read('metadata_inventory.json'):
    print(f"{r['form']:10} hits={r['metadata_rows']:6,} accessions={r['unique_accessions']:6,} primary={r['primary_document_rows']:6,} tracks={r['tracks']}")

425        hits=40,758 accessions=27,835 primary=27,731 tracks=['merger_exchange']
8-K        hits=49,227 accessions=31,232 primary=10,064 tracks=['conversion']
8-K/A      hits= 1,851 accessions= 1,172 primary=   308 tracks=['conversion']
DEFM14A    hits= 1,448 accessions= 1,443 primary= 1,443 tracks=['merger_exchange']
PREM14A    hits= 1,392 accessions=   861 primary=   861 tracks=['merger_exchange']
S-4        hits=17,478 accessions= 1,580 primary= 1,566 tracks=['merger_exchange']
S-4/A      hits=23,086 accessions= 3,113 primary= 3,079 tracks=['merger_exchange']
SC 13E3    hits= 1,698 accessions=   227 primary=   227 tracks=['merger_exchange']
SC 13E3/A  hits= 1,522 accessions=   798 primary=   798 tracks=['merger_exchange']


## Actual downloaded sample and structure

Five documents per form, chosen from the existing early-2020 convenience sample. Several share deals; no held-out performance claim is possible. HTML tables are counted but flattened; section/table-aware extraction is the next improvement.

In [3]:
for r in rows:
    print(f"{r['form']:10} {r['entity_name'][:35]:35} chars={r['characters']:9,} tables={r['html_tables']:4} {r['detection']}")

425        SRC Energy Inc.                     chars=   10,268 tables=   1 merger_candidate
PREM14A    SORL Auto Parts Inc                 chars=  648,162 tables= 394 merger_candidate
425        Steadfast Apartment REIT III, Inc.  chars=   10,399 tables=  15 review_no_strong_merger_cue
425        Steadfast Income REIT, Inc.         chars=    8,915 tables=   0 review_no_strong_merger_cue
425        Steadfast Apartment REIT III, Inc.  chars=   10,391 tables=  15 review_no_strong_merger_cue
DEFM14A    INSTRUCTURE INC                     chars=  828,630 tables= 418 merger_candidate
425        Steadfast Income REIT, Inc.         chars=   10,370 tables=  15 review_no_strong_merger_cue
SC 13E3    SORL Auto Parts Inc                 chars=   29,099 tables=  15 merger_candidate
DEFM14A    PB Bancorp, Inc.                    chars=  533,133 tables= 306 merger_candidate
SC 13E3/A  CRAFT BREW ALLIANCE, INC.           chars=   36,633 tables=   5 merger_candidate
DEFM14A    ANIXTER INTERNATIONAL INC

## Regex experiment

Compare all dollar mentions with contextual per-security candidates. The baseline excludes labelled par values and routes debt-exchange overlaps to review. Reduced counts do not imply measured precision. Dates have distinct roles; passage retrieval does not populate values.

In [4]:
comparison = read('method_comparison.json')
print('All dollar mentions:', sum(r['all_dollar_mentions'] for r in comparison))
print('Context cash candidates:', sum(r['context_cash_candidates'] for r in comparison))
print('Candidate fields:', summary['candidate_counts'])
print('Passage fields:', dict(Counter(p['field'] for r in rows for p in r['passages'])))

All dollar mentions: 9299
Context cash candidates: 354
Candidate fields: {'cash_amount': 354, 'conversion_ratio': 216, 'record_date': 11}
Passage fields: {'new_security_description': 4813, 'conditions': 515, 'expected_closing_date': 318, 'termination': 179, 'parties': 62, 'old_security_description': 286, 'default_option': 10, 'proration_terms': 18, 'available_options': 60}


## Inspected examples: cash, stock, and conditional terms

These assistant-inspected development examples help explain behavior. They are not independent human gold labels or final deal outcomes.

In [5]:
for e in read('spot_checks.json'):
    print('\n', e['entity_name'], e['field'], e['observed_value'])
    print(e['note'])
    print(e['url'])
    print(e['quote'])
    print('Candidate retrieved:', e['extractor_found_value'])


 INSTRUCTURE INC cash_amount 47.60
Proposed cash consideration in this filing; not eventual closing terms.
https://www.sec.gov/Archives/edgar/data/1355754/000119312520000113/d812272ddefm14a.htm
cease, with Instructure continuing as the surviving
corporation (the “Merger”) and a wholly owned subsidiary of Parent.
If the Merger is completed, you will be entitled to receive
$47.60 in cash, less any applicable withholding taxes, for each share of Instructure common stock that you own (unless you have properly exercised your appraisal rights).
The Board of Directors of Instructure (the “Board of Directors”), after considering the factors more fully described in
the enclosed proxy statement, has unanimously: (i) determined that it is in the best interests of Instructure and its stockholders, and declared it advisable,
Candidate retrieved: True

 TIFFANY & CO cash_amount 135.00
Proposed cash consideration; exclude par value and historical trading prices.
https://www.sec.gov/Archives/edgar/da

## Inspect a filing and its conflicting candidates

Change `entity_query` to examine another company. Repeated values are kept with distinct source offsets; conflicts are not automatically reconciled. Ranks are heuristic, not confidence probabilities.

In [6]:
entity_query = 'TIFFANY'
r = next(x for x in rows if entity_query.lower() in x['entity_name'].lower())
print(r['url'])
print(json.dumps(r['conflicting_values'], indent=2))
print(json.dumps(r['candidates'][:3], indent=2))

https://www.sec.gov/Archives/edgar/data/98246/000119312520001590/d841743ddefm14a.htm
{
  "cash_amount": [
    "120.00",
    "125.00",
    "130.00",
    "133.00",
    "133.85",
    "135.00",
    "91.70",
    "98.55"
  ]
}
[
  {
    "field": "cash_amount",
    "value": "135.00",
    "value_start": 45866,
    "value_end": 45872,
    "evidence_start": 45415,
    "evidence_end": 46330,
    "source_quote": "tatement.\n-5-\nTable of Contents\nOpinion of Goldman Sachs (Page 56)\nAt a meeting of the Board held on November 24, 2019, Goldman Sachs & Co. LLC (\u201cGoldman Sachs\u201d) rendered to the Board\nits oral opinion, subsequently confirmed in its written opinion dated November 24, 2019, to the effect that, as of the date of Goldman Sachs\u2019 written opinion, and based upon and subject to the factors and assumptions set forth in Goldman\nSachs\u2019 written opinion, the $135.00 in cash per share of common stock to be paid to the holders (other than Parent and its affiliates) of shares of

## Evidence integrity check

Offsets refer to saved normalized text, not raw HTML bytes. This verifies grounding locations, not semantic correctness.

In [7]:
checked = 0
for r in rows:
    text = (ROOT / r['normalized_path']).read_text()
    for c in r['candidates'] + r['passages']:
        assert text[c['evidence_start']:c['evidence_end']] == c['source_quote']
        checked += 1
print('Exact evidence spans checked:', checked)
assert all(v is None for r in rows for v in r['canonical_fields'].values())
print('All canonical values remain unverified/null.')

Exact evidence spans checked: 6842
All canonical values remain unverified/null.


## Evaluation and next step

Use `outputs/mergers/review_template.json` for independent labels, recording deal IDs and field evidence. Split by adjudicated deal, including all amendments and counterparties. Report detection F1, applicable critical-field accuracy, evidence support, latency, total cost, and escalation quality. EDGAR accession counts measure form retrieval, not semantic merger recall. LLM backward review supplements independent labels.

Next: collect merger-specific 8-Ks and filing bundles; resolve referenced agreements; segment summary/terms/background and preserve table cells; compare GLiNER or LLM review on the same held-out sample. Accounts, holdings and BNY response deadlines remain internal enrichment.

## Reproduce

From the merger_research folder, run `python scripts/run_merger_research.py` and `python -m unittest discover -s tests/mergers -v`. Run `python scripts/audit_merger_metadata.py` to recreate the metadata inventory. Downloads are optional and explicit. Existing project requirements provide BeautifulSoup/lxml/pandas/pyarrow. Re-running extraction preserves any existing review template.